In [1]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent 

import pandas as pd

# ============================================================
# 1. 원본 csv.gz 불러오기
# ============================================================

train = pd.read_csv(
    PROJECT_ROOT / 'data' / 'train.csv.gz'
)

test = pd.read_csv(
    PROJECT_ROOT / 'data' / 'test.csv.gz'
)


In [2]:
# ============================================================
# 2. timestamp를 datetime으로 변환
# ============================================================

train['timestamp'] = pd.to_datetime(
    train['timestamp'],
    errors='coerce'
)

test['timestamp'] = pd.to_datetime(
    test['timestamp'],
    errors='coerce'
)

# NaT 확인 확인
print('train timestamp 오류:', train['timestamp'].isnull().sum())
print('test timestamp 오류:', test['timestamp'].isnull().sum())

train timestamp 오류: 0
test timestamp 오류: 0


In [3]:
# ============================================================
# 3. row_id 인덱스 지정
# ============================================================

print('train row_id unique:', train['row_id'].is_unique)
print('test row_id unique:', test['row_id'].is_unique)
# 고유성 확인: True

# 인덱스 지정
train = train.set_index('row_id')
test = test.set_index('row_id')

train row_id unique: True
test row_id unique: True


### 파생변수 생성하기
#### 최종 파이프라인 기준
#### EDA, 분포비교, 이상치/결측치 처리 단계에서 행 제거 또는 데이터 변경이 있을경우
 - 행 제거가 있을경우 시계열파생변수는 필히 시계열 순서를 복원한 뒤에 다시 생성
 - 값 변동이 있을경우 파생변수도 수정 필요

In [4]:
# ============================================================
# 4. row_id 인덱스 지정
# ============================================================
# wl_diff_00h = wl_t(현재수위) - wl_t_minus_00h 파생변수 5개 생성

print('train 파생변수 추가 전 크기: ', train.shape)
print('test 파생변수 추가 전 크기: ', test.shape)

for h in [1, 3, 6, 12, 24]:
    train[f"wl_diff_{h}h"] = (
        train['wl_t'] - train[f"wl_t_minus_{h}h"]
    )
    test[f"wl_diff_{h}h"] = (
        test['wl_t'] - test[f"wl_t_minus_{h}h"]
    )

print('train 수위차분 파생변수 추가 후 크기: ', train.shape)
print('test 수위차분 파생변수 추가 후 크기: ', test.shape)

train 파생변수 추가 전 크기:  (749657, 25)
test 파생변수 추가 전 크기:  (186263, 24)
train 수위차분 파생변수 추가 후 크기:  (749657, 30)
test 수위차분 파생변수 추가 후 크기:  (186263, 29)


In [5]:
# ============================================================
# 5. 강우강도 파생변수 생성
# ============================================================
# imerg_int_0to0h = (imerg_rain_00h - imerg_rain_00h)/hours
# sfc_int_0to0h = (sfc_rain_00h - sfc_rain_00h)/hours

print('train 파생변수 추가 전 크기: ', train.shape)
print('test 파생변수 추가 전 크기: ', test.shape)


for df in [train, test]:

    df['imerg_int_1to6h'] = (df['imerg_rain_6h'] - df['imerg_rain_1h']) / 5
    df['imerg_int_6to24h'] = (df['imerg_rain_24h'] - df['imerg_rain_6h']) / 18

    df['sfc_int_1to6h'] = (df['sfc_rain_6h'] - df['sfc_rain_1h']) / 5
    df['sfc_int_6to24h'] = (df['sfc_rain_24h'] - df['sfc_rain_6h']) / 18


print('train 강우강도 파생변수 추가 후 크기: ', train.shape)
print('test 강우강도 파생변수 추가 후 크기: ', test.shape)

train 파생변수 추가 전 크기:  (749657, 30)
test 파생변수 추가 전 크기:  (186263, 29)
train 강우강도 파생변수 추가 후 크기:  (749657, 34)
test 강우강도 파생변수 추가 후 크기:  (186263, 33)


In [6]:
# ============================================================
# 4. train, test 데이터 셋 parquet 파일로 저장
# ============================================================
train.to_parquet(PROJECT_ROOT / 'data' / '1_train_full_fe.parquet')

test.to_parquet(PROJECT_ROOT / 'data' / '1_test_fe.parquet')

# 저장 완료 확인
print('Parquet 저장 완료')

Parquet 저장 완료
